In [ ]:
import numpy as np
from data import vars
from src.tracking.track import Track
from src.tracking.track_manager import TrackManager
from src.tracking.kalman_filter import KalmanFilter
import time
import matplotlib.pyplot as plt
from evaluation import metrics

In [ ]:
def measurement_noise(R):
    return np.random.multivariate_normal(mean=[0, 0], cov=vars.R).reshape(2, 1)

def angle_between(v1, v2):
    v1_u = v1 / np.linalg.norm(v1)
    v2_u = v2 / np.linalg.norm(v2)
    dot_product = np.dot(v1_u, v2_u)
    angle_rad = np.arccos(np.clip(dot_product, -1.0, 1.0))
    
    return np.degrees(angle_rad)

In [ ]:
true_state = []
P0 = np.diag([vars.R[0,0], vars.R[1,1], 100.0, 100.0])
truth_data = [ 
                {
        "id": 1,
        "x": np.array([[80.0],
                       [111.0],
                       [  3.0],
                       [ 0.0]], dtype=float),
        "P": P0
    },
    {
        "id": 2,
        "x": np.array([[25.0],
                       [111.0],
                       [  1.0],
                       [ -1.0]], dtype=float),
        "P": P0
    },
    {
        "id": 3,
        "x": np.array([[10.0],
                       [80.0],
                       [  1.0],
                       [ -1.0]], dtype=float),
        "P": P0
    },
    {
        "id": 4,
        "x": np.array([[20.0],
                       [140.0],
                       [  1.0],
                       [ 1.0]], dtype=float),
        "P": P0
    },
    {
        "id": 5,
        "x": np.array([[80.0],
                       [140.0],
                       [  -1.0],
                       [ 1.0]], dtype=float),
        "P": P0
    }
]
n_steps = 100
track_truths = [] # truth starts from k-1
measure_data = [] # measurements start from k
for state in truth_data:
    x_k = state['x'].copy()
    truth_states = {"id": state['id'], "x_states": [x_k.copy()], "P": P0}
    track_measurements = {"id": state['id']}
    for _ in range(n_steps):
        x_k = vars.F @ x_k
        truth_states['x_states'].append(x_k.copy())
        if not 'measurements' in track_measurements.keys():
            track_measurements['measurements'] = [vars.H @ x_k.copy() + measurement_noise(vars.R)]
        else:
            track_measurements['measurements'].append(vars.H @ x_k.copy() + measurement_noise(vars.R))
    track_truths.append(truth_states)
    measure_data.append(track_measurements)

truth_states = {}
for tid in track_truths: # truth starts from k-1 to k99 so you have 101 
    truth_states[tid['id']] =  [x for x in tid['x_states']]
truth_positions = {}
for tid in track_truths:
    truth_positions[tid['id']] =  [x[:2,:] for x in tid['x_states']]
truth_velocities = {}
for tid in track_truths:
    truth_velocities[tid['id']] =  [x[2:,:] for x in tid['x_states']]
truth_times = [i * vars.dt for i in range(101)] # k-1 to k_99 = 101
truth_exists = {}
for tid in track_truths:
    truth_exists[tid['id']] =  [1 for x in truth_times]
    if tid['id'] == 1:
        for i, val in enumerate(truth_exists[tid['id']]):
            if i in [10, 11, 12, 20, 30, 40, 51, 52]:
                truth_exists[tid['id']][i] = 0

In [ ]:
scans = {}
for i, (v, w, x, y, z) in enumerate(zip(*[md['measurements'] for md in measure_data])):
    scan_index = i + 1
    if scan_index in [10, 11, 12, 20, 30, 40, 51, 52]:
        scans[scan_index] = [w, x, y, z]
    else:
        scans[scan_index] = [v, w, x, y, z]

In [ ]:
# places all of the k-1 tracks into kf objects and track objects. 
trax = [
    Track(
        track['id'],
        KalmanFilter(F=vars.F, H=vars.H, R=vars.R, Q=vars.Q, x_hat_km1_km1=track["x"], P_km1_km1=track["P"])
    )
    for track in truth_data
]
tracker = TrackManager(trax, 5.991) # all tracks are stored into the trackmanager
# truth_states - For each track id x_i_t = [p_x, p_y, v_x, v_y] for i = {1,2,3,4,5} and t = {k-1, k, k+1, ... k+99}
# truth_positions - For each track id x_i_t = [p_x, p_y] for i = {1,2,3,4,5} and t = {k-1, k, k+1, ... k+99}
# truth_velocities - For each track id x_i_t = [v_x, v_y] for i = {1,2,3,4,5} and t = {k-1, k, k+1, ... k+99}
# truth_times - delta = 1.5, t = {delta*i, delta*i+1,...} for i = {0,1, 2, ... 100} 
# truth_exists - truth_exists = {1 if track_i should have a hit at t = k, 0 otherwise}
# scans - {z_1, z_2, z_3, ...} for z_i in F@x + v(0, R) for timestep k_j , j = {k, k+1, ...}
tracker.scan_log[0] = {"time": truth_times[0], "num_measurements": 0, 
                        "num_tracks": len(tracker.tracks)}

for index, measurements in scans.items():
    tracker.scan_log[index] = {"time": truth_times[index], 
                                "num_measurements": len(measurements), 
                                "num_tracks": len(tracker.tracks)}
    tracker.predict_all()
    assignments, unassigned_tracks, unassigned_measurements, _ = \
        tracker.gnn_associate(measurements, index)

    assignment_map = {a['track_id']: a['measurement'] for a in assignments}
    unassigned_track_ids = {t['track_id'] for t in unassigned_tracks}
    tracks_to_delete = []

    tracker.pred_log[index] = {}
    for trk in tracker.tracks:
        tracker.pred_log[index][trk.track_id] = [trk.kf.x_hat_k_km1, trk.kf.P_k_km1]
        
        if trk.track_id in assignment_map: 
            z_k = assignment_map[trk.track_id] # Get measurement from ass map

            trk.update(z_k) # use measurement / prediction to update 

            if trk.tentative: # if its a tentative track increase hit count by 1
                trk.hit_count += 1
                if trk.hit_count >= 2: # promote track if hit count >= 2
                    trk.promote_track()
                    print(f"tentative Track ID: {trk.track_id} Promoted! Hit Count: {trk.hit_count}")
            print(f"Track {trk.track_id} Detected!")

        elif trk.track_id in unassigned_track_ids: # if this track id is unassigned then mark as a miss and coast
                                                    # predication from k|k-1 -> k|k
            trk.miss()
            trk.coast()
            print(f"Track {trk.track_id} Missed Detection!")

        if trk.tentative and trk.missed_count >= 2: # delete a tentative track with missed count >= 2
            print(f"tentative Track ID: {trk.track_id} Deleted, Missed Count: {trk.missed_count} >= 2")
            tracks_to_delete.append(trk)

        elif (not trk.tentative) and trk.missed_count >= 3:
            # delete a confirmed track with miss count > = 3
            print(f"Confirmed Track ID: {trk.track_id} Deleted, Missed Count: {trk.missed_count} >= 3")
            tracks_to_delete.append(trk)

    for trk in tracks_to_delete: # accutally delete the tracks from the list.
        tracker.delete_track(trk)

    for meas in unassigned_measurements:
        # For each unassigned measurement create a tentative track
        ten_track_id = tracker.tentative_track(meas['measurement'], tracker.get_new_track_id())
        print(f"tentative Track ID: {ten_track_id} Created!")



In [ ]:
rmse, _, n = metrics.position_rmse_from_truth_and_predlog(truth_states, tracker.pred_log)
per_track_rmse = metrics.position_rmse_per_track_from_truth_and_predlog(truth_states, tracker.pred_log)
coverage = metrics.track_coverage_from_truth_and_predlog(truth_states, tracker.pred_log)
print("Total Tracks:", max([x.track_id for x in tracker.tracks]))
print("Overall RMSE:", rmse)
print("Per-track RMSE:", per_track_rmse)
print("Coverage:", coverage)
print("Matched samples:", n, "of", int(n/coverage))

In [ ]:
def sort_scan_keys(scan_keys):
    """
    Sort keys like:
    'k', 'k_1', 'k_2', 'k_3', ...
    """
    return sorted(scan_keys)

plt.figure(figsize=(8, 6))

# Plot truth trajectories
for track_id, pos_list in truth_positions.items():
    x_vals = [pos[0, 0] for pos in pos_list]
    y_vals = [pos[1, 0] for pos in pos_list]
    plt.plot(x_vals, y_vals, marker='o', label=f"Truth Track {track_id}")

# Collect predicted track positions across scans
pred_positions = {}

for scan in sort_scan_keys(tracker.pred_log.keys()):
    for track_id, track_data in tracker.pred_log[scan].items():
        state = track_data[0]   # [state_vector, covariance_matrix]
        x = state[0, 0]
        y = state[1, 0]

        if track_id not in pred_positions:
            pred_positions[track_id] = {"x": [], "y": []}

        pred_positions[track_id]["x"].append(x)
        pred_positions[track_id]["y"].append(y)

# Plot predicted trajectories
for track_id, vals in pred_positions.items():
    plt.plot(
        vals["x"],
        vals["y"],
        marker='x',
        linestyle='--',
        label=f"Pred Track {track_id}"
    )

plt.xlabel("X Position")
plt.ylabel("Y Position")
plt.title("Truth and Predicted Trajectories")
plt.legend()
plt.grid(True)
plt.axis("equal")
plt.show()